# 1. 첫 아군 사망 전 or 후 > 체력,쉴드,쿨다운 조건을 맞춰도 차이가 남는지 확인

### 결과 파일 읽기

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# 노트북을 저장소 어디에서 실행하든 프로젝트 최상위 경로를 찾는다.
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src/main.py").is_file()
)

# 분석할 실행을 명시적으로 선택한다. 다른 실행 결과를 섞지 않는다.
RUN_NAME = "30000_20261031-215640-258935"

ANALYSIS_DIR = (
    ROOT
    / "experiment_analysis/stage2_behavior_diagnostics/results/4s5z"
    / RUN_NAME
    / "analysis/window_5"
)

context = pd.read_csv(ANALYSIS_DIR / "context_comparison.csv")

display(context.head())

,outcome,unit_type,health_band,shield_band,cooldown_band,phase,valid_episodes,opportunities,nonattack_choices,episode_ratio_mean,episode_ratio_median,mean_ci95_low,mean_ci95_high
0,defeat,Stalker,0..0.3,0..0.3,ready,after_first_ally_death,18,30,1,0.018519,0.0,0.0,0.055556
1,defeat,Stalker,0..0.3,0..0.3,ready,before_first_ally_death,4,4,0,0.000000,0.0,0.0,0.000000
2,defeat,Stalker,0..0.3,0..0.3,waiting,after_first_ally_death,24,217,4,0.014836,0.0,0.0,0.041035
3,defeat,Stalker,0..0.3,0..0.3,waiting,before_first_ally_death,17,30,0,0.000000,0.0,0.0,0.000000
4,defeat,Stalker,0.3..0.7,0..0.3,ready,after_first_ally_death,18,36,2,0.022222,0.0,0.0,0.066667


### 첫 아군 사망 전 > 승리, 패배만 선택하기

In [5]:
# 시간 초과는 이번 비교에서 분리한다.
# 초반 행동을 보려는 것이므로 첫 아군 사망 전 구간만 선택한다.
early = context.loc[
    context["outcome"].isin(["win", "defeat"])
    & context["phase"].eq("after_first_ally_death") # before_first_ally_death
].copy()

display(early)

,outcome,unit_type,health_band,shield_band,cooldown_band,phase,valid_episodes,opportunities,nonattack_choices,episode_ratio_mean,episode_ratio_median,mean_ci95_low,mean_ci95_high
0,defeat,Stalker,0..0.3,0..0.3,ready,after_first_ally_death,18,30,1,0.018519,0.000000,0.000000,0.055556
2,defeat,Stalker,0..0.3,0..0.3,waiting,after_first_ally_death,24,217,4,0.014836,0.000000,0.000000,0.041035
4,defeat,Stalker,0.3..0.7,0..0.3,ready,after_first_ally_death,18,36,2,0.022222,0.000000,0.000000,0.066667
6,defeat,Stalker,0.3..0.7,0..0.3,waiting,after_first_ally_death,24,278,3,0.006579,0.000000,0.000000,0.019737
8,defeat,Stalker,0.7..1,0..0.3,ready,after_first_ally_death,19,46,0,0.000000,0.000000,0.000000,0.000000
10,defeat,Stalker,0.7..1,0..0.3,waiting,after_first_ally_death,24,389,0,0.000000,0.000000,0.000000,0.000000
12,defeat,Stalker,0.7..1,0.3..0.7,ready,after_first_ally_death,21,44,0,0.000000,0.000000,0.000000,0.000000
14,defeat,Stalker,0.7..1,0.3..0.7,waiting,after_first_ally_death,24,280,0,0.000000,0.000000,0.000000,0.000000
16,defeat,Stalker,0.7..1,0.7..1,ready,after_first_ally_death,24,225,0,0.000000,0.000000,0.000000,0.000000
18,defeat,Stalker,0.7..1,0.7..1,waiting,after_first_ally_death,24,1588,4,0.002357,0.000000,0.000000,0.005601


### 같은 조건의 승리, 패배를 나란히 비교하기

In [6]:
# 아래 네 조건이 모두 같은 경우끼리 비교한다.
CONDITIONS = [
    "unit_type",
    "health_band",
    "shield_band",
    "cooldown_band",
]

# 같은 조건의 win과 defeat를 서로 다른 열로 배치한다.
wide = early.pivot(
    index=CONDITIONS,
    columns="outcome",
    values=[
        "episode_ratio_mean",
        "valid_episodes",
        "opportunities",
    ],
)

# 기존 CSV의 '에피소드별 비율의 평균'을 사용한다.
# 전체 행동 횟수를 합쳐 비율을 다시 계산하면 에피소드의 가중치가 달라진다.
comparison = pd.DataFrame({
    "win_pct": wide["episode_ratio_mean"]["win"] * 100,
    "defeat_pct": wide["episode_ratio_mean"]["defeat"] * 100,
    "win_episodes": wide["valid_episodes"]["win"],
    "defeat_episodes": wide["valid_episodes"]["defeat"],
    "win_opportunities": wide["opportunities"]["win"],
    "defeat_opportunities": wide["opportunities"]["defeat"],
}).reset_index()

# 음수이면 같은 조건에서도 패배에서 비공격 선택이 적었다는 뜻이다.
comparison["gap_pp"] = (
    comparison["defeat_pct"] - comparison["win_pct"]
)

# 양쪽에 최소 5판이 있는 조건을 먼저 살펴보기 위한 표시다.
# 5판은 탐색용 기준이며, 통계적 유의성을 보장하는 기준은 아니다.
MIN_EPISODES = 5

comparison["sample_ok"] = (
    comparison["win_episodes"].ge(MIN_EPISODES)
    & comparison["defeat_episodes"].ge(MIN_EPISODES)
)

# 표본 조건을 충족한 행부터, 패배에서 비공격 선택이 더 적은 순서로 표시한다.
comparison = comparison.sort_values(
    ["sample_ok", "gap_pp"],
    ascending=[False, True],
)

display(comparison.round(2))

,unit_type,health_band,shield_band,cooldown_band,win_pct,defeat_pct,win_episodes,defeat_episodes,win_opportunities,defeat_opportunities,gap_pp,sample_ok
0,Stalker,0..0.3,0..0.3,ready,16.22,1.85,25.0,18.0,86.0,30.0,-14.37,True
14,Zealot,0.3..0.7,0..0.3,ready,10.98,0.00,48.0,17.0,520.0,88.0,-10.98,True
10,Zealot,0..0.3,0..0.3,ready,43.14,34.55,48.0,16.0,470.0,87.0,-8.59,True
20,Zealot,0.7..1,0.3..0.7,ready,8.55,0.00,13.0,6.0,45.0,14.0,-8.55,True
4,Stalker,0.7..1,0..0.3,ready,6.16,0.00,50.0,19.0,203.0,46.0,-6.16,True
1,Stalker,0..0.3,0..0.3,waiting,5.46,1.48,40.0,24.0,404.0,217.0,-3.97,True
11,Zealot,0..0.3,0..0.3,waiting,10.10,6.48,55.0,24.0,1169.0,326.0,-3.62,True
8,Stalker,0.7..1,0.7..1,ready,3.59,0.00,55.0,24.0,543.0,225.0,-3.59,True
6,Stalker,0.7..1,0.3..0.7,ready,3.03,0.00,47.0,21.0,148.0,44.0,-3.03,True
15,Zealot,0.3..0.7,0..0.3,waiting,2.63,0.38,55.0,24.0,1702.0,550.0,-2.25,True


### 관심 있는 조건 하나만 자세히 보기

In [7]:
# 체력·실드가 70%를 넘고, 무기 쿨다운이 남아 있는 Stalker
example = comparison.loc[
    comparison["unit_type"].eq("Stalker")
    & comparison["health_band"].eq("0.7..1")
    & comparison["shield_band"].eq("0.7..1")
    & comparison["cooldown_band"].eq("waiting")
]

display(example.round(2))

,unit_type,health_band,shield_band,cooldown_band,win_pct,defeat_pct,win_episodes,defeat_episodes,win_opportunities,defeat_opportunities,gap_pp,sample_ok
9,Stalker,0.7..1,0.7..1,waiting,1.65,0.24,55.0,24.0,3816.0,1588.0,-1.41,True
